<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session9_Automated_Turing_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0

In [2]:
from flint import acb, arb, arb_series, ctx

from fractions import Fraction
import csv
import json
import time

ctx.prec = 192

T_LO = arb("10")
T_HI = arb("1000")

print("SESSION 09 INITIALIZED")
print("FLINT precision:", ctx.prec, "bits")
print("Height range: 10 to 1000")

SESSION 09 INITIALIZED
FLINT precision: 192 bits
Height range: 10 to 1000


In [3]:
#Rigorous total zero count using Turing's method
print("TURING ZERO COUNT")
print("-----------------")

start = time.time()

N_LO = T_LO.zeta_nzeros()
N_HI = T_HI.zeta_nzeros()

n_lo = N_LO.unique_fmpz()
n_hi = N_HI.unique_fmpz()

if n_lo is None or n_hi is None:
    raise RuntimeError("Could not isolate endpoint zero counts.")

n_lo = int(n_lo)
n_hi = int(n_hi)

total_zeros = n_hi - n_lo

print("N(10):", n_lo)
print("N(1000):", n_hi)
print("Zeros between heights:", total_zeros)

print("Elapsed seconds:", round(time.time() - start, 3))

assert total_zeros == 649

TURING ZERO COUNT
-----------------
N(10): 0
N(1000): 649
Zeros between heights: 649
Elapsed seconds: 0.003


In [6]:
#Generate the rigorously isolated zeros
print("GENERATING CERTIFIED ZERO INTERVALS")
print("------------------------------------")

start = time.time()

first_zero_index = n_lo + 1

zeros = list(
    acb.zeta_zeros(first_zero_index, total_zeros)
)

print("Zeros returned:", len(zeros))
print("Expected:", total_zeros)

assert len(zeros) == total_zeros

print("\nFIRST FIVE ZERO ENCLOSURES")

for i, zero in enumerate(zeros[:5], start=1):
    print(i, zero)

print("\nLAST FIVE ZERO ENCLOSURES")

for i, zero in enumerate(zeros[-5:], start=total_zeros - 4):
    print(i, zero)

print("\nElapsed seconds:", round(time.time() - start, 3))

GENERATING CERTIFIED ZERO INTERVALS
------------------------------------
Zeros returned: 649
Expected: 649

FIRST FIVE ZERO ENCLOSURES
1 0.500000000000000000000000000000000000000000000000000000000 + [14.1347251417346937904572519835624702707842571156992431757 +/- 1.77e-56]j
2 0.500000000000000000000000000000000000000000000000000000000 + [21.0220396387715549926284795938969027773343405249027817546 +/- 3.24e-56]j
3 0.500000000000000000000000000000000000000000000000000000000 + [25.0108575801456887632137909925628218186595496725579966725 +/- 1.31e-56]j
4 0.500000000000000000000000000000000000000000000000000000000 + [30.4248761258595132103118975305840913201815600237154401810 +/- 4.65e-56]j
5 0.500000000000000000000000000000000000000000000000000000000 + [32.9350615877391896906623689640749034888127156035170390093 +/- 3.80e-56]j

LAST FIVE ZERO ENCLOSURES
645 0.500000000000000000000000000000000000000000000000000000000 + [994.404590571094491900027805466234876673672244208449361438 +/- 2.16e-55]j
64

In [7]:
#Create our own disjoint rational intervals
radius = Fraction(1, 100)

def rational_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


records = []

for index, zero in enumerate(zeros, start=first_zero_index):

    gamma = zero.imag

    # Use the certified midpoint only to choose a convenient interval.
    center = Fraction(
        gamma.mid().str(55, radius=False)
    )

    left = center - radius
    right = center + radius

    left_ball = rational_ball(left)
    right_ball = rational_ball(right)

    # Check the complete zero-ordinate enclosure is inside the window.
    contained = (
        left_ball.upper() < gamma.lower()
        and gamma.upper() < right_ball.lower()
    )

    if not contained:
        raise RuntimeError(
            f"Zero {index}: interval does not contain its full enclosure."
        )

    records.append({
        "index": index,
        "zero": zero,
        "gamma": gamma,
        "left": left,
        "right": right
    })


# Verify all intervals are mutually disjoint.
for i in range(len(records) - 1):

    if not records[i]["right"] < records[i + 1]["left"]:
        raise RuntimeError(
            f"Intervals {i+1} and {i+2} overlap."
        )

print("Intervals generated:", len(records))
print("All zero enclosures contained:", True)
print("All intervals disjoint:", True)

Intervals generated: 649
All zero enclosures contained: True
All intervals disjoint: True


In [9]:
#Independent endpoint sign audit
def hardy_z(t_fraction):

    t = rational_ball(t_fraction)

    result = arb_series(
        [t, 1],
        prec=2
    ).riemann_siegel_z()

    return result.coeffs()[0]


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verification_rows = []
verified = 0
failed = []

start = time.time()

print("HARDY Z SIGN VERIFICATION")
print("-------------------------")

for i, record in enumerate(records, start=1):

    z_left = hardy_z(record["left"])
    z_right = hardy_z(record["right"])

    sign_left = certified_sign(z_left)
    sign_right = certified_sign(z_right)

    passed = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    if passed:
        verified += 1
    else:
        failed.append(record["index"])

    verification_rows.append({
        "zero_index": record["index"],
        "left": str(record["left"]),
        "right": str(record["right"]),
        "left_sign": sign_left,
        "right_sign": sign_right,
        "passed": passed
    })

    if i % 100 == 0 or i == len(records):
        print(
            f"Checked {i}/{len(records)} | "
            f"Passed: {verified} | "
            f"Failed: {len(failed)}"
        )

print("\nTOTAL PASSED:", verified)
print("TOTAL FAILED:", len(failed))
print("Elapsed seconds:", round(time.time() - start, 3))

if failed:
    print("Failed zero indices:", failed)

HARDY Z SIGN VERIFICATION
-------------------------
Checked 100/649 | Passed: 100 | Failed: 0
Checked 200/649 | Passed: 200 | Failed: 0
Checked 300/649 | Passed: 300 | Failed: 0
Checked 400/649 | Passed: 400 | Failed: 0
Checked 500/649 | Passed: 500 | Failed: 0
Checked 600/649 | Passed: 600 | Failed: 0
Checked 649/649 | Passed: 649 | Failed: 0

TOTAL PASSED: 649
TOTAL FAILED: 0
Elapsed seconds: 1.5


In [10]:
SESSION08_CONTOUR_COUNT = 649

print("METHOD COMPARISON")
print("-----------------")

print("Session 08 contour count:", SESSION08_CONTOUR_COUNT)
print("Session 09 Turing count:", total_zeros)
print("Session 09 isolated zeros:", len(records))
print("Session 09 sign changes:", verified)

matches = (
    SESSION08_CONTOUR_COUNT == total_zeros
    and total_zeros == len(records)
    and verified == total_zeros
)

print("\nALL RESULTS AGREE:", matches)

METHOD COMPARISON
-----------------
Session 08 contour count: 649
Session 09 Turing count: 649
Session 09 isolated zeros: 649
Session 09 sign changes: 649

ALL RESULTS AGREE: True


In [11]:
with open(
    "Session09_zero_intervals.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zero_index",
            "left",
            "right",
            "left_sign",
            "right_sign",
            "passed"
        ]
    )

    writer.writeheader()
    writer.writerows(verification_rows)


summary = {
    "session": 9,
    "method": "FLINT Turing method",
    "height_range": [10, 1000],
    "precision_bits": 192,
    "N_10": n_lo,
    "N_1000": n_hi,
    "total_zeros": total_zeros,
    "isolated_zeros": len(records),
    "hardy_sign_changes": verified,
    "failed_intervals": failed,
    "session08_contour_count": SESSION08_CONTOUR_COUNT,
    "all_checks_passed": matches
}

with open(
    "Session09_verification_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)


print("SESSION 09 FINAL REPORT")
print("=======================")
print(json.dumps(summary, indent=2))

if matches:
    print("\nSESSION 09: SUCCESS")
else:
    print("\nSESSION 09: NOT YET COMPLETE")

SESSION 09 FINAL REPORT
{
  "session": 9,
  "method": "FLINT Turing method",
  "height_range": [
    10,
    1000
  ],
  "precision_bits": 192,
  "N_10": 0,
  "N_1000": 649,
  "total_zeros": 649,
  "isolated_zeros": 649,
  "hardy_sign_changes": 649,
  "failed_intervals": [],
  "session08_contour_count": 649,
  "all_checks_passed": true
}

SESSION 09: SUCCESS
